# E13 — Siamese đa pha + encoder ResNet18 nạp MedicalNet, ở ĐỦ độ phân giải

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

Tự tải trọng số từ HuggingFace. **Bật `Internet` trong Notebook options.** Cần mount
**cache E4** (không phải cache E12).

## Vì sao đây không phải một thí nghiệm lặt vặt nữa

E6, E6b, E12, TTA đều null. Cả bốn đụng vào augmentation hoặc thủ thuật lúc suy luận, mà
bảng có kiểm soát của văn liệu nói rõ đó là những biến đáng 1–2 điểm. Đòn bẩy lớn nằm ở
**cách hợp nhất 8 thì**, và dự án chưa từng thử nó ở điều kiện công bằng.

Bốn bằng chứng độc lập chỉ về cùng một chỗ:

| bằng chứng | con số |
|---|---|
| Bảng SDR-Former: ResNet-50 early-fusion → SNN-UniFormer-S | 0.6898 → **0.7639** (+0.074) |
| Đội hạng 2 challenge dùng ResNet18 | 0.8078 — backbone không phải nút thắt |
| Mọi phương pháp > 0.75 trong bảng CGHNet | đều xử lý **từng thì** rồi mới hợp nhất |
| Hai lớp yếu của ta | ICC và di căn, hai lớp chẩn đoán bằng **động học ngấm thuốc** |

Ghép 8 thì thành 8 kênh của conv đầu là trộn chúng ngay ở lớp 1, trước khi mạng học được
đặc trưng nào để so sánh **giữa** các thì. Đó là giả thuyết cơ chế cho việc ICC và di căn
mãi không lên.

## Ràng buộc số học, đọc trước khi kỳ vọng

Từ đúng số test-104 (ICC 0.519 · di căn 0.273):

| nếu 5 lớp còn lại đều đạt | trần macro-F1 |
|---|---|
| 0.85 | 0.720 |
| 0.90 | **0.756** |
| 0.95 | 0.792 |

**Giữ nguyên hai lớp đó thì kể cả 5 lớp kia hoàn hảo cũng chỉ tới 0.827.** Nên khi đọc kết
quả, F1 của ICC và di căn quan trọng ngang macro-F1.

## Vì sao E2 thất bại, và vì sao lần này khác

E2 chạy Siamese ở **48×48 in-plane** trong khi văn liệu dùng 112–128, vì DenseNet121-3D hạ
mẫu 5 lần và cần ≥32 mọi chiều, buộc `input_downsample: [2,2,1]`. WORKLOG S-065 kết luận:
biến gây nhiễu độ phân giải là thủ phạm, **không phải ý tưởng Siamese**.

ResNet-3D hạ mẫu 16 lần rồi adaptive-pool nên chạy được ở đủ 112×112×32 với
`input_downsample: [1,1,1]`. **Cổng B đo trực tiếp điều này** — E2 chết âm thầm, lần này
không.

## Cộng hưởng thật giữa Siamese và pretrained

Trọng số MedicalNet dành cho ảnh **một kênh**. Ở chế độ Siamese encoder cũng nhìn đúng một
thì mỗi lượt, nên `conv1` được dùng **đúng như lúc nó được học**. E8 (early-concat) phải
nhân bản conv đầu ra 8 kênh rồi chia cho 8. Và encoder chạy `feed_forward=False` nên không
có `fc`, tức **không khoá nào được phép thiếu** — cổng khớp trọng số chặt hơn E8.

## 🎯 Phép cô lập sạch nhất dự án từng có

E8 và E13 khác nhau **đúng một thứ**: early-concat so với Siamese. Cùng backbone, cùng file
trọng số, cùng cache, cùng loss, cùng recipe.

    E13 − E8  =  hiệu ứng của riêng wrapper Siamese

So trực tiếp được với +0.074 của SDR-Former.

## Bar quyết định — CHỐT TRƯỚC KHI CHẠY

Sàng fold 1+2 (162 ca). Mốc E4 trên đúng 162 ca đó: **0.6879**.

| fold 1+2 | kết luận |
|---|---|
| **≥ 0.79** | 0.75 trên test-104 còn khả thi → chạy đủ 5 fold |
| 0.72 – 0.79 | tiến bộ thật nhưng **không tới 0.75**. Chạy 5 fold để có số, và nói rõ mục tiêu không đạt |
| 0.69 – 0.72 | ngang E4. Fusion cũng không phải đòn bẩy → **dừng** |
| < 0.69 | lỗi triển khai, không phải kết luận khoa học. Soi cổng B |

Bar là 0.79 vì để test-104 đạt 0.75 thì out-of-fold phải **~0.82** (thiên lệch chọn epoch
đã đo = −0.069). Bar thấp hơn chỉ là tự lừa mình.

⚠️ Dương trên 2 fold **chỉ đủ để "chưa loại được"**. E6b sàng 2 fold cho +0.038 rồi 5 fold
cho −0.002.

## 0. Bootstrap

Dòng `repo commit` là bằng chứng đang chạy đúng bản code nào.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

# ---- THAM SỐ ---------------------------------------------------------------
FOLDS = [1, 2]                            # sàng 2 fold; đủ 5 fold mới kết luận
CONFIG_NAME = "e13_siamese_pretrained.yaml"
SCOPE = "model."                          # khối được phép đổi
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

EXPERIMENT = Path(CONFIG_NAME).stem
os.environ["LLDMMRI_OUTPUT_DIR"] = f"/kaggle/working/runs/{EXPERIMENT}"
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

CFG_PATH = REPO / "configs" / CONFIG_NAME
CFG = load_yaml(CFG_PATH)
M = CFG["model"]

print(f"\nthí nghiệm: {EXPERIMENT} · fold {FOLDS}")
print(f"fusion  : {M['name']} · {M['fusion']} · phase_embedding={M['phase_embedding']}")
print(f"encoder : {M['encoder']}{M['encoder_depth']} · shortcut {M['shortcut_type']} · "
      f"bias_downsample {M['bias_downsample']} · conv1_stride {M['conv1_stride']}")
print(f"input_downsample: {M['input_downsample']}   <- [1,1,1] = KHÔNG hạ mẫu (chỗ E2 chết)")

## Cổng 0 ⚠️ — config này khác baseline ở ĐÚNG những chỗ nào

E13 là phép đổi **kiến trúc**. Một khoá lọt sang `data.` hay `train.` biến nó thành phép
đổi hai cụm biến, và sai đó không để lại dấu vết nào trong kết quả.

In [ ]:
BASE = load_yaml(REPO / "configs" / "baseline_3dpatch.yaml")


def flatten(d, prefix=""):
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out.update(flatten(v, prefix + k + "."))
        else:
            out[prefix + k] = v
    return out


fa, fb = flatten(BASE), flatten(CFG)
diff = {
    k: (fa.get(k), fb.get(k))
    for k in sorted(set(fa) | set(fb))
    if str(fa.get(k)) != str(fb.get(k))
}
MIEN = ("output_dir", "fold")

print(f"{CONFIG_NAME} khác baseline ở {len(diff)} khoá:")
for k, (a, b) in diff.items():
    ghi_chu = "   (miễn)" if k in MIEN else ("" if k.startswith(SCOPE) else "   <-- NGOÀI")
    print(f"  {k}: {a!r} -> {b!r}{ghi_chu}")

ngoai = [k for k in diff if not k.startswith(SCOPE) and k not in MIEN]
assert not ngoai, f"⛔ khác biệt NGOÀI {SCOPE}: {ngoai} — không còn là phép so kiến trúc"

# Recipe train phải trùng KHÍT baseline, không chỉ "nằm ngoài scope".
for khoi in ("train", "loss", "data"):
    assert CFG[khoi] == BASE[khoi], f"⛔ khối {khoi} lệch baseline"
print(f"\n✓ chỉ khác trong {SCOPE} · train/loss/data trùng khít baseline")

## 1. Trọng số MedicalNet

Ba nguồn, thử theo thứ tự: đã mount → đã tải trong session này → tải từ HuggingFace
`TencentMedicalNet/MedicalNet-Resnet18`, file `resnet_18_23dataset.pth` (132 MB).

File tải về nằm ở `/kaggle/working/weights` nên **Save Version** giữ lại được.

In [ ]:
MEDICALNET_FILE = "resnet_18_23dataset.pth"
HF_REPO = "TencentMedicalNet/MedicalNet-Resnet18"
MIN_BYTES = 100 * 1024**2

WEIGHTS_DIR = Path("/kaggle/working/weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
dest = WEIGHTS_DIR / MEDICALNET_FILE

mounted = [p for p in sorted(Path("/kaggle/input").rglob("resnet_18*.pth"))]
if mounted:
    WEIGHTS = mounted[0]
    print(f"dùng bản đã mount: {WEIGHTS}")
elif dest.exists() and dest.stat().st_size >= MIN_BYTES:
    WEIGHTS = dest
    print(f"dùng bản đã tải trong session này: {WEIGHTS}")
else:
    print(f"tải từ HuggingFace: {HF_REPO}/{MEDICALNET_FILE} ...")
    url = f"https://huggingface.co/{HF_REPO}/resolve/main/{MEDICALNET_FILE}"
    try:
        from huggingface_hub import hf_hub_download

        WEIGHTS = Path(hf_hub_download(
            repo_id=HF_REPO, filename=MEDICALNET_FILE, local_dir=str(WEIGHTS_DIR)
        ))
    except Exception as exc:  # noqa: BLE001 - hạ cấp sang urllib rồi báo lỗi tử tế
        print(f"  huggingface_hub không dùng được ({type(exc).__name__}), thử urllib")
        import urllib.error
        import urllib.request

        try:
            urllib.request.urlretrieve(url, dest)
            WEIGHTS = dest
        except (urllib.error.URLError, OSError) as exc2:
            raise RuntimeError(
                f"Không tải được trọng số ({exc2}).\n"
                "  Gần như chắc chắn Internet đang TẮT. Bật ở Notebook options rồi chạy\n"
                f"  lại cell này. Nếu tài khoản chưa xác minh điện thoại thì tải tay từ\n"
                f"    {url}\n"
                "  rồi upload thành Kaggle Dataset và mount vào; cell này sẽ tự nhận."
            ) from None

size = WEIGHTS.stat().st_size
assert size >= MIN_BYTES, (
    f"{WEIGHTS} chỉ {size / 1e6:.1f} MB, cần ~132 MB. Nhiều khả năng là trang lỗi HTML "
    "được lưu thành .pth. Xoá file và chạy lại."
)

os.environ["LLDMMRI_PRETRAINED_PATH"] = str(WEIGHTS)
print(f"\ntrọng số: {WEIGHTS}  ({size / 1e6:.0f} MB)")

## Cổng A ⚠️⚠️ — trọng số vào đúng encoder dùng chung, và conv đầu KHÔNG bị nhân bản

Chế độ hỏng của pretrained là **im lặng**: model vẫn dựng, vẫn train, vẫn ra số, chỉ là
một phần trọng số ngẫu nhiên.

Ba phép kiểm, và phép thứ ba là điểm khác biệt so với E8:

1. `build_model` không nổ — cặp `shortcut_type`/`bias_downsample` khớp biến thể sinh ra
   file, và **không khoá nào thiếu** (encoder không có `fc` vì `feed_forward=False`).
2. Trùng khớp **bit-exact** với file.
3. **`conv1.in_channels == 1`.** Đây là chỗ Siamese hơn early-concat: trọng số MedicalNet
   là một kênh, và ở đây nó được dùng nguyên vẹn. E8 phải nhân bản ra 8 kênh rồi chia 8.

In [ ]:
import torch

from src.models import build_model
from src.models.densenet3d import count_parameters

model = build_model(CFG["model"])
print(f"tham số: {count_parameters(model):,}")

enc = model.encoder
assert enc.conv1.in_channels == 1, (
    f"conv đầu nhận {enc.conv1.in_channels} kênh — đang chạy early-concat, không phải Siamese"
)
assert enc.fc is None, "encoder vẫn còn lớp fc; cần feed_forward=False để lấy đặc trưng"
print(f"conv1.in_channels = {enc.conv1.in_channels} · encoder.fc = {enc.fc} · "
      f"embed_dim = {model.embed_dim}")

raw = torch.load(WEIGHTS, map_location="cpu")
state = raw.get("state_dict", raw) if isinstance(raw, dict) else raw
state = {k.replace("module.", "", 1): v for k, v in state.items()}
sd = enc.state_dict()

bit_exact = [
    k for k, v in state.items()
    if k in sd and v.shape == sd[k].shape and torch.equal(v, sd[k])
]
print(f"khoá trùng khớp bit-exact với file: {len(bit_exact)}/{len(sd)}")
assert len(bit_exact) == len(sd), (
    "encoder một kênh thì MỌI khoá phải đến từ file, không có ngoại lệ nào. "
    f"Thiếu: {sorted(set(sd) - set(bit_exact))[:10]}"
)
print("✓ toàn bộ encoder đến từ trọng số MedicalNet, conv đầu dùng NGUYÊN VẸN")

## 2. Cache E4

Nhận diện bằng **nội dung** `cache_meta.json`, không bằng tên dataset.

⚠️ **Phải loại cache E12.** Nó cũng `per_phase` + `lesion_tight` + `target_size`
112×112×32 nên ba khoá thường dùng không phân biệt được. Khác biệt là `crop_margin_voxels`;
cho nhầm thì model nhận khối 136×136×40 và **không có gì báo lỗi**.

In [ ]:
import json as _json

E4_KEYS = {
    "align_phases": "per_phase",
    "crop_mode": "lesion_tight",
    "target_size": [112, 112, 32],
}
GRID = (8, 112, 112, 32)

ung_vien, CACHE_DIR = [], None
for meta_path in sorted(Path("/kaggle/input").rglob("cache_meta.json")):
    try:
        meta = _json.loads(meta_path.read_text("utf-8"))
    except Exception:  # noqa: BLE001 - chỉ để liệt kê chẩn đoán
        continue
    le = meta.get("crop_margin_voxels")
    khop = all(meta.get(k) == v for k, v in E4_KEYS.items()) and not any(le or [])
    ung_vien.append((meta_path.parent, meta, khop))
    if khop and CACHE_DIR is None:
        CACHE_DIR = meta_path.parent

print(f"=== {len(ung_vien)} cache tìm thấy dưới /kaggle/input ===")
for path, meta, khop in ung_vien:
    print(f"  {'✓ E4 ' if khop else '  -- '}  {path}")
    print(f"          size={meta.get('target_size')} lề={meta.get('crop_margin_voxels')} "
          f"align={meta.get('align_phases')} crop={meta.get('crop_mode')}")

if CACHE_DIR is None:
    raise RuntimeError(
        "Chưa mount cache E4.\n"
        f"  Cần cache có {E4_KEYS} và KHÔNG có crop_margin_voxels.\n"
        "  Cache E12 (lề 12/12/4, mảng 136x136x40) KHÔNG dùng được cho config này."
    )

os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)

import numpy as np

meta = _json.loads((CACHE_DIR / "cache_meta.json").read_text("utf-8"))
n_npz = len(list(CACHE_DIR.glob("*.npz")))
assert n_npz >= 498, f"chỉ có {n_npz} ca, cần 498"
mau = next(CACHE_DIR.glob("*.npz"))
with np.load(mau) as z:
    shape = tuple(z["image"].shape)
assert shape == GRID, f"hình dạng {shape}, cần {GRID} — đây không phải cache E4"
print(f"\ncache E4 ✓ · {CACHE_DIR} · {n_npz} ca · mảng {shape}")

## Cổng B ⚠️⚠️⚠️ — encoder có nhận ĐỦ 112×112×32 không

**Cổng quan trọng nhất của cả notebook.** Đây đúng là chỗ E2 chết, và nó chết âm thầm: E2
chạy ở 48×48 in-plane suốt 100+ epoch, cho 0.35–0.49, và phải mất một phiên đọc paper mới
phát hiện nguyên nhân không phải Siamese mà là độ phân giải (WORKLOG S-065).

Cell này gắn forward hook lên encoder và **đo hình dạng thật** đi vào nó, thay vì tin vào
config. Kỳ vọng:

    vào encoder : (B*8, 1, 112, 112, 32)     <- 1 kênh, đủ độ phân giải
    ra encoder  : (B*8, 512)                 <- feed_forward=False có tác dụng
    trọng số thì: (B, 8), tổng theo trục thì = 1

In [ ]:
from src.train.run import build_loaders

train_loader, val_loader, _ = build_loaders(CFG, FOLDS[0])
batch = next(iter(train_loader))
x = batch["image"]
B, P = x.shape[0], x.shape[1]
print(f"batch từ loader: {tuple(x.shape)}")

assert isinstance(model.pre_pool, torch.nn.Identity), (
    "pre_pool KHÔNG phải Identity -> encoder đang bị hạ mẫu. Đây đúng là lỗi của E2. "
    f"input_downsample = {CFG['model']['input_downsample']}"
)

seen = {}
handle = model.encoder.register_forward_hook(
    lambda m, i, o: seen.update(vao=tuple(i[0].shape), ra=tuple(o.shape))
)
# Chạy trên GPU: 16 mẫu 112x112x32 qua conv1 stride 1 trên 4 vCPU thì chờ khá lâu.
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(dev).eval()
with torch.no_grad():
    logits = model(x.to(dev))
handle.remove()

print(f"vào encoder : {seen['vao']}")
print(f"ra encoder  : {seen['ra']}")
print(f"logits      : {tuple(logits.shape)}")

assert seen["vao"] == (B * P, 1, *x.shape[2:]), (
    f"encoder nhận {seen['vao']}, cần {(B * P, 1, *tuple(x.shape[2:]))}"
)
assert seen["vao"][2:] == (112, 112, 32), (
    f"⛔ encoder chỉ nhận {seen['vao'][2:]} thay vì (112, 112, 32) — ĐÚNG LỖI CỦA E2. "
    "Đừng train, kiểm input_downsample."
)
assert seen["ra"] == (B * P, model.embed_dim)
assert tuple(logits.shape) == (B, CFG["model"]["num_classes"])

w = model.last_phase_weights.cpu()
assert w is not None and tuple(w.shape) == (B, P)
torch.testing.assert_close(w.sum(dim=1), torch.ones(B))
print(f"\ntrọng số thì lúc khởi tạo: {[round(v, 4) for v in w[0].tolist()]}")
print("  (khởi tạo thì ~đều là bình thường; cổng D kiểm lại SAU khi train)")
print("\n✓ Siamese chạy ở ĐỦ độ phân giải — khác hẳn E2")

## Cổng C ⚠️ — ngân sách. 8 lượt forward là rủi ro thật

E4 (DenseNet, early-concat): GPU ~20 s/epoch, tổng 45 s/epoch, 3.76 h/fold.
E8 (ResNet18, early-concat): GPU **18** s/epoch, CPU 41 s/epoch, 3.4 h/fold.

E13 chạy encoder **8 lượt** thay vì 1, nên GPU gần chắc chắn thành nút thắt mới.

| GPU đo được | làm gì |
|---|---|
| dưới ~45 s/epoch | vẫn CPU chặn, ngân sách như E4. Chạy tiếp |
| 45–70 s/epoch | GPU chặn, ~5–6 h/fold. Vẫn chạy được 2 fold trong một session |
| trên ~70 s/epoch | đặt `model.conv1_stride: [1, 2, 2]` trong config, commit, chạy lại từ bootstrap. Hạ mẫu trong mặt phẳng như Med3D và giữ nguyên z, rẻ hơn ~4 lần |

In [ ]:
import time

model = model.train()   # `dev` và model đã lên GPU ở cổng B
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler("cuda")

xb = x.to(dev)
yb = torch.zeros(B, dtype=torch.long, device=dev)


def buoc():
    opt.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=torch.float16):
        out = model(xb)
        loss = torch.nn.functional.cross_entropy(out, yb)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    return out


for _ in range(3):
    buoc()
torch.cuda.synchronize()
t0 = time.time()
for _ in range(15):
    buoc()
torch.cuda.synchronize()
giay_batch = (time.time() - t0) / 15

so_batch = len(train_loader)
t0, n = time.time(), 0
for _ in train_loader:
    n += 1
    if n >= 20:
        break
cpu_epoch = (time.time() - t0) / n * so_batch
gpu_epoch = giay_batch * so_batch
epoch = max(cpu_epoch, gpu_epoch)
gio_fold = epoch * int(CFG["train"]["epochs"]) / 3600

print(f"VRAM đỉnh   : {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
print(f"GPU fwd+bwd : {gpu_epoch:6.0f} s/epoch  ({so_batch} batch × {giay_batch:.3f}s)")
print(f"CPU nạp+aug : {cpu_epoch:6.0f} s/epoch")
print(f"ước tính    : {epoch:6.0f} s/epoch -> {gio_fold:.1f} h/fold -> "
      f"{gio_fold * len(FOLDS):.1f} h cho {len(FOLDS)} fold")
print("\n(E8 ResNet18 early-concat: GPU 18 s/epoch, CPU 41, 3.4 h/fold)")
if gpu_epoch > 70:
    print("\n⚠ GPU trên 70 s/epoch — cân nhắc conv1_stride: [1, 2, 2] (xem markdown trên)")
if gio_fold * len(FOLDS) > 11.0:
    print("\n⚠ vượt 11h — bỏ bớt fold khỏi FOLDS")

del model, opt, xb, yb
torch.cuda.empty_cache()

## 3. Train

`resume: true` nên bị ngắt giữa chừng thì chạy lại đúng cell này, nó đọc tiếp từ `last.pt`.

⚠️ `train` đọc YAML **từ đĩa**, không dùng biến `CFG` trong notebook. Sửa `CFG` bằng tay ở
cell nào đó sẽ không có tác dụng lúc train, mà cổng 0 lại đọc từ chính file nên vẫn báo
xanh. Muốn đổi tham số thì sửa file config trong repo rồi chạy lại từ bootstrap.

In [ ]:
import time

from src.train.run import train

assert os.environ.get("LLDMMRI_PRETRAINED_PATH"), "env trọng số trống — chạy lại mục 1"
assert os.environ.get("LLDMMRI_CACHE_DIR"), "env cache trống — chạy lại mục 2"

t0 = time.time()
results = {}

for fold in FOLDS:
    print("\n" + "=" * 60)
    print("FOLD %d  (da dung %.2fh)" % (fold, (time.time() - t0) / 3600))
    print("=" * 60)
    results[fold] = train(CFG_PATH, fold_override=fold)
    print("fold %d xong: macro-F1 %.4f" % (fold, results[fold]["macro_f1"]))

print("\ntong: %.2f h" % ((time.time() - t0) / 3600))

## 4. Kết quả, đối chiếu bar đã chốt

Nhắc lại bar: fold 1+2 gộp **≥ 0.79** thì 0.75 trên test-104 còn khả thi; **0.69–0.72** là
ngang E4 và nên dừng.

⚠️ Chênh lệch từng fold là nhiễu nếu nhìn riêng (CI mỗi fold ~±0.19).

In [ ]:
import csv as _csv

OUT = Path(os.environ["LLDMMRI_OUTPUT_DIR"])
E4 = {1: 0.7001, 2: 0.6771, 3: 0.7304, 4: 0.6680, 5: 0.6618}
E4_N = {1: 82, 2: 80, 3: 78, 4: 77, 5: 77}
E4_DAY = {1: 100, 2: 79, 3: 227, 4: 3, 5: 14}

print(f"{'fold':>5}{'epoch':>7}{'macro-F1':>11}{'kappa':>9}{'E4':>9}{'hiệu':>9}")
print("-" * 50)
rows, tong_n, tong_f1, tong_e4 = [], 0, 0.0, 0.0
for d in sorted(OUT.glob("fold*")):
    f = d / "metrics_best.json"
    if not f.exists():
        continue
    m = _json.loads(f.read_text("utf-8"))
    fold = int(m["fold"])
    rows.append((d, fold, m))
    n = E4_N[fold]
    tong_n += n
    tong_f1 += n * m["macro_f1"]
    tong_e4 += n * E4[fold]
    print(f"{fold:>5}{m['epoch']:>7}{m['macro_f1']:>11.4f}{m['cohen_kappa']:>9.4f}"
          f"{E4[fold]:>9.4f}{m['macro_f1'] - E4[fold]:>+9.4f}")

if tong_n:
    tb, tb_e4 = tong_f1 / tong_n, tong_e4 / tong_n
    print("-" * 50)
    print(f"{'TB':>5}{'':>7}{tb:>11.4f}{'':>9}{tb_e4:>9.4f}{tb - tb_e4:>+9.4f}")
    print()
    if tb >= 0.79:
        print(f"=> {tb:.4f} >= 0.79 : mục tiêu 0.75 trên test-104 CÒN KHẢ THI. Chạy đủ 5 fold.")
    elif tb >= 0.72:
        print(f"=> {tb:.4f} trong [0.72, 0.79) : tiến bộ THẬT nhưng KHÔNG tới 0.75.")
        print("   Chạy 5 fold để có số báo cáo, và nói rõ mục tiêu không đạt.")
    elif tb >= 0.69:
        print(f"=> {tb:.4f} trong [0.69, 0.72) : ngang E4. Fusion cũng không phải đòn bẩy.")
        print("   DỪNG. 0.75 nằm ngoài ngân sách còn lại.")
    else:
        print(f"=> {tb:.4f} < 0.69 : nghi LỖI TRIỂN KHAI, không phải kết luận khoa học.")
        print("   Soi lại cổng B (hình dạng vào encoder) và cổng A (khớp trọng số).")

print("\nEpoch chạm đáy val_loss — dự báo gần trọn vẹn F1 cuối (ρ=0.770, S-107):")
for d, fold, _m in rows:
    log = d / "train_log.csv"
    if log.exists():
        vl = [float(r["val_loss"]) for r in _csv.DictReader(open(log))]
        print(f"  fold {fold}: đáy @ epoch {vl.index(min(vl)) + 1:>3}   (E4: {E4_DAY[fold]})")

print("\nF1 từng lớp — ICC và di căn là hai lớp CHẶN mục tiêu về số học:")
for d, fold, m in rows:
    pc = m.get("per_class_f1")
    if pc:
        from src.data.taxonomy import SHORT_NAMES

        print(f"  fold {fold}: " + " · ".join(
            f"{SHORT_NAMES[i]} {v:.3f}" for i, v in enumerate(pc)
        ))

## Cổng D ⚠️ — phase attention có học được gì không

Cổng duy nhất nói về **cơ chế** thay vì về điểm số. Nếu trọng số 8 thì đều xấp xỉ
`1/8 = 0.125` thì fusion không học được gì, và Siamese chỉ là một cách lấy trung bình đắt
gấp 8 lần — khi đó dù macro-F1 có nhúc nhích cũng không phải nhờ lý do ta nghĩ.

Đối chiếu với độ nhạy theo thì đo bằng Grad-CAM trên 4 ca (WORKLOG S-098): **In Phase và
Out Phase thấp nhất ở cả 4 ca** (0.043–0.092, đều dưới mức đều 0.125), còn các thì có
thuốc thì cao. Nếu attention học đúng thì nó nên đi cùng chiều.

In [ ]:
PHASES = ["C-pre", "C+A", "C+V", "C+Delay", "T2WI", "DWI", "In Phase", "Out Phase"]
FOLD_D = FOLDS[0]

ckpt_dir = next(OUT.glob(f"fold{FOLD_D}_*"))
state = torch.load(ckpt_dir / "best.pt", map_location="cpu")
model_d = build_model(CFG["model"])
model_d.load_state_dict(state["model"])
model_d = model_d.to(dev).eval()

_, val_d, _ = build_loaders(CFG, FOLD_D)
tong = torch.zeros(8)
dem = 0
with torch.no_grad():
    for b in val_d:
        model_d(b["image"].to(dev))
        w = model_d.last_phase_weights.cpu()
        tong += w.sum(dim=0)
        dem += w.shape[0]
tb_w = (tong / dem).tolist()

print(f"fold {FOLD_D} · epoch {state['epoch']} · {dem} ca val")
print(f"{'thì':<12}{'trọng số':>10}{'so mức đều':>13}")
print("-" * 35)
for name, v in zip(PHASES, tb_w):
    print(f"{name:<12}{v:>10.4f}{v - 0.125:>+13.4f}")

trai = max(tb_w) - min(tb_w)
print(f"\ntrải (max - min): {trai:.4f}")
if trai < 0.02:
    print("⚠ SUY BIẾN — 8 thì gần như bằng nhau. Fusion không học được gì; Siamese ở đây")
    print("  chỉ là một cách lấy trung bình đắt gấp 8. Điểm số có tăng cũng không phải")
    print("  nhờ cơ chế ta nghĩ, và phải nói điều đó trong báo cáo.")
else:
    print("✓ attention có phân biệt giữa các thì.")
    print("  Đối chiếu Grad-CAM S-098: In/Out Phase nên ở nhóm THẤP nhất.")

del model_d
torch.cuda.empty_cache()

## 5. Gói mang về

In [ ]:
import shutil

PACK = Path(f"/kaggle/working/{EXPERIMENT}_results")
shutil.rmtree(PACK, ignore_errors=True)
PACK.mkdir(parents=True)

KEEP = ["val_probs_best.npz", "val_probs_last.npz", "metrics_best.json",
        "train_log.csv", "config_used.json", "best.pt"]
for d in sorted(OUT.glob("fold*")):
    dst = PACK / d.name
    dst.mkdir(parents=True, exist_ok=True)
    for name in KEEP:
        if (d / name).exists():
            shutil.copy2(d / name, dst / name)

total = sum(f.stat().st_size for f in PACK.rglob("*") if f.is_file())
print(f"đã gói {PACK}: {total / 2**20:.1f} MiB")
if WEIGHTS.is_relative_to(Path("/kaggle/working")):
    print(f"trọng số giữ ở {WEIGHTS.parent} — Save Version rồi mount cho session sau")

print("""
⚠ TẢI VỀ: giải nén CHỈ MỘT LỚP. File .npz và .pt bản thân là zip; trình giải nén bung
  đệ quy sẽ biến chúng thành thư mục và src.eval.* không thấy (đã dính, S-078).

Ở local, đặt vào runs/E13/ rồi:
    python -m src.eval.compare --baseline runs/E4_cv_results --candidate runs/E13
    python -m src.eval.compare --baseline runs/E8           --candidate runs/E13
                                          ^ cô lập ĐÚNG hiệu ứng wrapper Siamese
""")